# 03 · Budget checks (proof of the matched budget)
Every method calls one `BudgetedEvaluator`, which counts every fit and refuses fit 257.
A cheap stand-in objective is used here so the check takes seconds; the counting is the same as with real models.

In [1]:
import os
from pathlib import Path
import pandas as pd
from IPython.display import display
from hpo import config, data, spaces, starts, methods, runner, stats, plots

pd.set_option("display.precision", 5)
# Results folder: the full run by default. Set HPO_RESULTS=smoke to read the smoke test.
RESULTS = (
    config.SMOKE_DIR if os.environ.get("HPO_RESULTS") == "smoke" else config.RESULTS_DIR
)
FIGS, TABLES = RESULTS / "figures", RESULTS / "tables"
TABLES.mkdir(parents=True, exist_ok=True)
print("Reading/writing:", RESULTS.relative_to(config.ROOT))

Reading/writing: results


In [2]:
import numpy as np
import mealpy
from hpo.evaluator import BudgetedEvaluator

print("mealpy", mealpy.__version__)
cheap = lambda u: -float(np.sum((np.asarray(u) - 0.3) ** 2))

mealpy 3.0.2


## Each method × model uses exactly 256 fits

In [3]:
rows = []
for m in config.MODELS:
    for me in config.METHODS:
        ev = BudgetedEvaluator(m, None, fitness=cheap)
        methods.run(me, ev, seed=7)
        first16 = np.array([r["u"] for r in ev.trace[:16]])
        same_start = (
            np.allclose(first16, starts.shared_starts(7, spaces.dims(m)), atol=1e-6)
            if me != "grid"
            else None
        )
        rows.append(
            {
                "model": m,
                "method": me,
                "fits": ev.count,
                "first_16_are_shared_starts": same_start,
            }
        )
check = pd.DataFrame(rows)
display(check.pivot(index="method", columns="model", values="fits"))
assert (check["fits"] == config.BUDGET).all()
assert check.dropna()["first_16_are_shared_starts"].all()
print(
    "PASS: every method used exactly",
    config.BUDGET,
    "fits; Random/GA/DE/PSO started from the shared starts",
)

model,rf,svm,xgb
method,,,
de,256,256,256
ga,256,256,256
grid,256,256,256
pso,256,256,256
random,256,256,256


PASS: every method used exactly 256 fits; Random/GA/DE/PSO started from the shared starts


## The guard also stops a method that would run longer

In [4]:
for me in ["ga", "de", "pso"]:
    ev = BudgetedEvaluator("xgb", None, budget=20, fitness=cheap)
    methods.run(me, ev, seed=0)
    print(me, "stopped at", ev.count)

ga stopped at 20
de stopped at 20
pso stopped at 20


## Curves start together: best-so-far after fit 16 is identical for Random, GA, DE, PSO

In [5]:
vals = {}
for me in ["random", "ga", "de", "pso"]:
    ev = BudgetedEvaluator("svm", None, fitness=cheap)
    methods.run(me, ev, seed=5)
    vals[me] = ev.trace[15]["best_so_far"]
print(vals)
assert len(set(vals.values())) == 1

{'random': -0.027912267249945586, 'ga': -0.027912267249945586, 'de': -0.027912267249945586, 'pso': -0.027912267249945586}
